# NB21 수수료 시장 — 21장

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB21-1: EIP-1559 기본 수수료 갱신 — 참조 구현의 정수 산술 그대로. 런던 포크 첫 블록으로 검산한 뒤 혼잡 시나리오를 돌린다 (공부용)
BASE_FEE_MAX_CHANGE_DENOMINATOR = 8      # EIP-1559 원문 상수
ELASTICITY_MULTIPLIER = 2
INITIAL_BASE_FEE = 1_000_000_000          # 1 gwei (wei 단위)

def next_base_fee(parent_base_fee: int, parent_gas_used: int, parent_gas_limit: int) -> int:
    """부모 블록만의 함수(러프가든의 착상 (2)). 나눗셈은 전부 정수 나눗셈."""
    target = parent_gas_limit // ELASTICITY_MULTIPLIER
    if parent_gas_used == target:
        return parent_base_fee
    if parent_gas_used > target:
        delta = parent_base_fee * (parent_gas_used - target) // target // BASE_FEE_MAX_CHANGE_DENOMINATOR
        return parent_base_fee + max(delta, 1)
    delta = parent_base_fee * (target - parent_gas_used) // target // BASE_FEE_MAX_CHANGE_DENOMINATOR
    return parent_base_fee - delta

# 검산 1: 런던 포크 첫 블록 12,965,000 (2021-08-05T12:33:42Z) → 12,965,001. 헤더 값은 공개 RPC 두 곳의 원시 응답(각주)
b_12965000, used, limit = INITIAL_BASE_FEE, 30_025_257, 30_029_122
calc = next_base_fee(b_12965000, used, limit)
print(f"블록 12,965,000: 기본 수수료 {b_12965000:,} wei, 가스 {used:,}/{limit:,} (목표 {limit // 2:,})")
print(f"계산한 12,965,001의 기본 수수료 {calc:,} wei — 체인에 적힌 값 1,124,967,822 과 일치: {calc == 1_124_967_822}")

# 검산 2: 꽉 찬 블록은 정확히 +12.5%, 빈 블록은 −12.5%
print(f"꽉 찬 블록 뒤: {next_base_fee(10**9, 60_000_000, 60_000_000) / 10**9:.4f} gwei, 빈 블록 뒤: {next_base_fee(10**9, 0, 60_000_000) / 10**9:.4f} gwei")

# 시나리오: 기준일 관측 가스 한도 60,000,000(목표 30,000,000)에서 혼잡(20블록 꽉 참) → 해소(20블록 목표의 1/4) → 정상(목표)
limit = 60_000_000                        # 2026-10-24 관측값(상수 아님, 부록 A-10)
scenario = [limit] * 20 + [limit // 8] * 20 + [limit // 2] * 5
b = 10**9
print(f"\n{'블록':>4} {'가스 사용':>12} {'기본 수수료(gwei)':>18} {'변화':>8}")
for i, g in enumerate(scenario, 1):
    nb = next_base_fee(b, g, limit)
    if i <= 3 or i in (10, 20, 21, 22, 23, 30, 40, 41, 45):
        print(f"{i:>4} {g:>12,} {nb / 10**9:>18.4f} {(nb / b - 1):>+8.2%}")
    b = nb
print("꽉 찬 블록이 20개 이어지면 기본 수수료는 (1.125)^20 ≈ {:.1f}배가 된다. 수요가 줄면 내려온다(빈 블록이면 블록마다 12.5%씩).".format(1.125 ** 20))

블록 12,965,000: 기본 수수료 1,000,000,000 wei, 가스 30,025,257/30,029,122 (목표 15,014,561)
계산한 12,965,001의 기본 수수료 1,124,967,822 wei — 체인에 적힌 값 1,124,967,822 과 일치: True
꽉 찬 블록 뒤: 1.1250 gwei, 빈 블록 뒤: 0.8750 gwei

  블록        가스 사용       기본 수수료(gwei)       변화
   1   60,000,000             1.1250  +12.50%
   2   60,000,000             1.2656  +12.50%
   3   60,000,000             1.4238  +12.50%
  10   60,000,000             3.2473  +12.50%
  20   60,000,000            10.5451  +12.50%
  21    7,500,000             9.5565   -9.37%
  22    7,500,000             8.6606   -9.37%
  23    7,500,000             7.8486   -9.37%
  30    7,500,000             3.9403   -9.37%
  40    7,500,000             1.4723   -9.37%
  41   30,000,000             1.4723   +0.00%
  45   30,000,000             1.4723   +0.00%
꽉 찬 블록이 20개 이어지면 기본 수수료는 (1.125)^20 ≈ 10.5배가 된다. 수요가 줄면 내려온다(빈 블록이면 블록마다 12.5%씩).


In [ ]:
# NB21-2: 1차 가격 경매 vs 1559 — 같은 사용자들, 같은 블록 공간. 「남이 얼마를 쓸까」 추측이 지불에 어떻게 들어오는지 (공부용)
# 평가액 v 는 가스당 지불 의사(단위 없는 모형값). 블록 용량 C = 4 거래, 전부 같은 크기 1. 채굴자 한계비용 μ = 0.
users = {"A": 10.0, "B": 8.0, "C": 7.5, "D": 7.0, "E": 6.0, "F": 3.0}      # 평가액(진실)
C = 4

def fpa(bids):
    """1차 가격 경매: 높은 입찰 C 개 포함, 각자 자기 입찰액을 내고 전부 채굴자에게."""
    order = sorted(bids.items(), key=lambda kv: -kv[1])[:C]
    return {u: b for u, b in order}, sum(b for _, b in order), 0.0

def mech1559(valuations, base_fee, tip=0.01):
    """1559: 상한 = 평가액, 팁 = μ 근처의 작은 값(정리 5.7의 「명백한 최적 입찰」). 기본 수수료는 소각, 팁만 채굴자에게."""
    eligible = sorted(((u, v) for u, v in valuations.items() if v >= base_fee), key=lambda kv: -kv[1])[:C]
    paid = {u: min(base_fee + tip, v) for u, v in eligible}
    burned = base_fee * len(eligible)
    return paid, sum(paid.values()) - burned, burned

print("(가) FPA — 사용자가 진실한 평가액을 그대로 입찰하면 과다 지불이 생긴다")
paid, miner, burned = fpa(users)
print(f"   포함: {paid}  채굴자 수입 {miner:.2f}, 소각 0")
print("   A 는 10 을 냈지만 6 을 조금 넘게만 냈어도 들어갔다(탈락 최고 입찰 E = 6) — 그래서 FPA 에서는 평가액대로 입찰하는 것이 우월전략이 아니다(DSIC 아님)")

print("\n(가′) FPA — 모두가 「탈락 최고 입찰 근처」로 깎아 부르는 균형 비슷한 상태를 흉내 내면")
shaded = {u: round(v * 0.8, 2) for u, v in users.items()}
paid, miner, burned = fpa(shaded)
print(f"   포함: {paid}  채굴자 수입 {miner:.2f} — 깎는 폭은 남의 입찰에 대한 추측에 달려 있다")

for base_fee in [6.5, 7.2, 3.0]:
    paid, miner, burned = mech1559(users, base_fee)
    label = "적정" if base_fee == 6.5 else ("높음" if base_fee == 7.2 else "지나치게 낮음")
    print(f"\n(나) 1559 — 기본 수수료 {base_fee} ({label}): 포함 {paid}")
    n_elig = sum(1 for v in users.values() if v >= base_fee)
    if n_elig > C:
        print(f"   소각 {burned:.2f}, 채굴자 수입(팁) {miner:.2f}. 적격자 {n_elig}명 > 용량 {C} — 이 코드는 팁 경쟁을 계산하지 않는다(정리 5.7의 예외 자리, 아래 줄).")
    else:
        print(f"   소각 {burned:.2f}, 채굴자 수입(팁) {miner:.2f}. 사용자는 남의 입찰을 추측할 필요가 없다(상한 = 평가액).")
print("\n기본 수수료 3.0 처럼 지나치게 낮으면 기본 수수료를 낼 사용자가 용량보다 많아 팁 경쟁이 되살아난다 — 1559 가 FPA 로 되돌아가는 자리(정리 5.7의 조건).")

(가) FPA — 사용자가 진실한 평가액을 그대로 입찰하면 과다 지불이 생긴다
   포함: {'A': 10.0, 'B': 8.0, 'C': 7.5, 'D': 7.0}  채굴자 수입 32.50, 소각 0
   A 는 10 을 냈지만 6 을 조금 넘게만 냈어도 들어갔다(탈락 최고 입찰 E = 6) — 그래서 FPA 에서는 평가액대로 입찰하는 것이 우월전략이 아니다(DSIC 아님)

(가′) FPA — 모두가 「탈락 최고 입찰 근처」로 깎아 부르는 균형 비슷한 상태를 흉내 내면
   포함: {'A': 8.0, 'B': 6.4, 'C': 6.0, 'D': 5.6}  채굴자 수입 26.00 — 깎는 폭은 남의 입찰에 대한 추측에 달려 있다

(나) 1559 — 기본 수수료 6.5 (적정): 포함 {'A': 6.51, 'B': 6.51, 'C': 6.51, 'D': 6.51}
   소각 26.00, 채굴자 수입(팁) 0.04. 사용자는 남의 입찰을 추측할 필요가 없다(상한 = 평가액).

(나) 1559 — 기본 수수료 7.2 (높음): 포함 {'A': 7.21, 'B': 7.21, 'C': 7.21}
   소각 21.60, 채굴자 수입(팁) 0.03. 사용자는 남의 입찰을 추측할 필요가 없다(상한 = 평가액).

(나) 1559 — 기본 수수료 3.0 (지나치게 낮음): 포함 {'A': 3.01, 'B': 3.01, 'C': 3.01, 'D': 3.01}
   소각 12.00, 채굴자 수입(팁) 0.04. 적격자 6명 > 용량 4 — 이 코드는 팁 경쟁을 계산하지 않는다(정리 5.7의 예외 자리, 아래 줄).

기본 수수료 3.0 처럼 지나치게 낮으면 기본 수수료를 낼 사용자가 용량보다 많아 팁 경쟁이 되살아난다 — 1559 가 FPA 로 되돌아가는 자리(정리 5.7의 조건).


In [ ]:
# NB21-3: EIP-4844 블롭 가스 기본 수수료 — 원문의 fake_exponential 과 상수를 그대로 옮긴다 (공부용)
MAX_BLOB_GAS_PER_BLOCK = 786_432          # 블롭 6개
TARGET_BLOB_GAS_PER_BLOCK = 393_216       # 블롭 3개
MIN_BASE_FEE_PER_BLOB_GAS = 1             # wei
BLOB_BASE_FEE_UPDATE_FRACTION = 3_338_477
GAS_PER_BLOB = 2**17

def fake_exponential(factor: int, numerator: int, denominator: int) -> int:
    """EIP-4844 Helpers: factor * e**(numerator/denominator) 의 테일러 전개 정수 근사(원문 그대로)."""
    i = 1
    output = 0
    numerator_accum = factor * denominator
    while numerator_accum > 0:
        output += numerator_accum
        numerator_accum = (numerator_accum * numerator) // (denominator * i)
        i += 1
    return output // denominator

def calc_excess_blob_gas(parent_excess: int, parent_blob_gas_used: int) -> int:
    if parent_excess + parent_blob_gas_used < TARGET_BLOB_GAS_PER_BLOCK:
        return 0
    return parent_excess + parent_blob_gas_used - TARGET_BLOB_GAS_PER_BLOCK

def base_fee_per_blob_gas(excess: int) -> int:
    return fake_exponential(MIN_BASE_FEE_PER_BLOB_GAS, excess, BLOB_BASE_FEE_UPDATE_FRACTION)

import math
print(f"상한까지 찬 블록(6블롭, 목표는 3) 하나당 변화율: e^(393216/3338477) = {math.exp(TARGET_BLOB_GAS_PER_BLOCK / BLOB_BASE_FEE_UPDATE_FRACTION):.4f}")
# 시나리오: 30블록 연속 6블롭(상한) → 30블록 0블롭 → 목표(3블롭)
excess = 0
print(f"{'블록':>4} {'블롭 수':>6} {'excess_blob_gas':>16} {'블롭 가스당 기본 수수료(wei)':>26}")
for i, blobs in enumerate([6] * 30 + [0] * 30 + [3] * 3, 1):
    fee = base_fee_per_blob_gas(excess)
    if i in (1, 2, 3, 10, 20, 30, 31, 32, 40, 50, 60, 61, 63):
        print(f"{i:>4} {blobs:>6} {excess:>16,} {fee:>26,}")
    excess = calc_excess_blob_gas(excess, blobs * GAS_PER_BLOB)
print("상한 블록 30개 뒤 초과분은 30 × 393,216 = {:,} 이고 수수료는 e^(그 값/3,338,477) ≈ {:.0f}배. 빈 블록 30개면 초과분은 0 으로 돌아와 최소 1 wei 가 된다.".format(30 * 393_216, math.exp(30 * 393_216 / 3_338_477)))

상한까지 찬 블록(6블롭, 목표는 3) 하나당 변화율: e^(393216/3338477) = 1.1250
  블록   블롭 수  excess_blob_gas         블롭 가스당 기본 수수료(wei)
   1      6                0                          1
   2      6          393,216                          1
   3      6          786,432                          1
  10      6        3,538,944                          2
  20      6        7,471,104                          9
  30      6       11,403,264                         30
  31      0       11,796,480                         34
  32      0       11,403,264                         30
  40      0        8,257,536                         11
  50      0        4,325,376                          3
  60      0          393,216                          1
  61      3                0                          1
  63      3                0                          1
상한 블록 30개 뒤 초과분은 30 × 393,216 = 11,796,480 이고 수수료는 e^(그 값/3,338,477) ≈ 34배. 빈 블록 30개면 초과분은 0 으로 돌아와 최소 1 wei 가 된다.
